# Prompt History Viewer

Run the cells from top to bottom to browse prompts stored in `studio.db`.
No extra packages are required beyond the active Jupyter kernel.


## 1. Load recent generations

Change `LIMIT` if you want more or fewer rows. The database is opened read-only.


In [ ]:
import html
import json
import sqlite3
from IPython.display import HTML, display

LIMIT = 50
connection = sqlite3.connect("file:studio.db?mode=ro", uri=True)
connection.row_factory = sqlite3.Row
rows = connection.execute(
    """
    SELECT created_at, id AS generation_id, parent_id, moodboard_id,
           is_baseline, seed, aspect_ratio,
           resolution_width || 'x' || resolution_height AS resolution,
           negative_prompt, COALESCE(NULLIF(compiled_prompt, ''), prompt) AS compiled_prompt,
           schema_json
    FROM generations
    ORDER BY created_at DESC
    LIMIT ?
    """,
    (LIMIT,),
).fetchall()

columns = ("created_at", "generation_id", "seed", "aspect_ratio", "resolution", "is_baseline")
table = "<table><tr>" + "".join(f"<th>{name}</th>" for name in columns) + "</tr>"
table += "".join("<tr>" + "".join(f"<td>{html.escape(str(row[name] or ''))}</td>" for name in columns) + "</tr>" for row in rows)
display(HTML(table + "</table>"))


## 2. Show full details

The newest generation is selected automatically. Replace the ID below with any ID from the table.


In [ ]:
GENERATION_ID = rows[0]["generation_id"] if rows else None
record = next((row for row in rows if row["generation_id"] == GENERATION_ID), None)
if record is None:
    raise ValueError(f"Generation not found in the latest {LIMIT} rows: {GENERATION_ID}")

try:
    scene_json = json.dumps(json.loads(record["schema_json"] or "{}"), indent=2, ensure_ascii=False)
except json.JSONDecodeError:
    scene_json = record["schema_json"] or ""

def section(title, value):
    display(HTML(f"<h3>{html.escape(title)}</h3><pre style='white-space:pre-wrap'>{html.escape(str(value or ''))}</pre>"))

section("Metadata", "\n".join(f"{key}: {record[key]}" for key in ("created_at", "generation_id", "parent_id", "moodboard_id", "seed", "aspect_ratio", "resolution", "is_baseline")))
section("Negative prompt", record["negative_prompt"])
section("Compiled prompt submitted to the API", record["compiled_prompt"])
section("Scene JSON", scene_json)


## Tips

- Change `GENERATION_ID` and rerun the detail cell to inspect another request.
- Increase `LIMIT` if an older ID is not in the summary.
- Keep the notebook beside `studio.db`; otherwise update the database path in the first cell.


The query uses SQLite read-only mode, so running this notebook cannot modify prompt history.
